In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_A = spark.createDataFrame([('XY',34,'ABJHG'),('OP',49,'KLJHG'),('WE',74,'OUTEW'),('OP',92,'KLJHG')], schema = ['A','B','C'])
df_B = spark.createDataFrame([('XY','ABJHG','HSUWJ'),('OP','KLLHG','GZUWZ'),('WE','OUTEW','HSJWS')], schema = ['D','E','F'])

## hash key generate

In [0]:

df_A = df_A.withColumn('A_HASH', md5(concat_ws('_',col('A'), col('C'))))
df_A.display()

In [0]:

df_B = df_B.withColumn('B_HASH', md5(concat_ws('_',col('D'), col('E'))))
df_B.display()

In [0]:
df_A.join(
    df_B, on=[(df_A["A"] == df_B["D"]) & (df_A["C"] == df_B["E"])], how="inner"
).display()

In [0]:
df_inner = df_A.join(df_B,on=[(df_A['A_HASH']==df_B['B_HASH'])],how='inner')
df_inner.explain()
df_inner.select('A','B','C','D','E','F').display()

In [0]:
df_joined = df_A.join(
    df_B, on=[(df_A["A"] == df_B["D"]) & (df_A["C"] == df_B["E"])], how="left"
)
df_joined.display()

In [0]:
df_inner.explain()

In [0]:
df_joined.explain()

In [0]:
df_A.join(
    df_B, on=[(df_A["A"] == df_B["D"]) & (df_A["C"] == df_B["E"])], how="right"
).display()

In [0]:
df_A.join(
    df_B, on=[(df_A["A"] == df_B["D"]) & (df_A["C"] == df_B["E"])], how="left_semi"
).display()

In [0]:
df_A.join(
    df_B, on=[(df_A["A"] == df_B["D"]) & (df_A["C"] == df_B["E"])], how="left_anti"
).display()

In [0]:
df_A.createOrReplaceTempView('DF_A')
df_B.createOrReplaceTempView('DF_B')

In [0]:
# Global temporary views are not supported on serverless compute
# Use regular temporary views instead (createOrReplaceTempView)
# df_A.createOrReplaceGlobalTempView('DF_A_glob')
# df_B.createOrReplaceGlobalTempView('DF_B_glob')

In [0]:
%sql
select * from DF_A where A='XY'

In [0]:
%sql

select DF_A.a,DF_A.B,DF_A.C,DF_B.D,Df_B.E from DF_A inner join DF_B
on DF_A.A=DF_B.D and DF_A.C=DF_B.E

In [0]:
perform join operation on Employee and dept data...
Inner join left right outer...left_semi, left_anti
--using spark and sql (creating view)

## Repartiton  & coalesce

In [0]:
df_flight = spark.read.format('csv').option('header', True).load('/Volumes/workspace/default/datasets/Flight_data.csv')
df_Indigo = df_flight.filter(lower(col('Airline'))=='indigo')

In [0]:
# df_flight.display()

In [0]:
df_flight.count()

In [0]:
#df_flight.rdd.map(lambda row: len(str(row))).sum()

In [0]:
df_flight = df_flight.withColumn('Partition_id', spark_partition_id())

In [0]:
df_flight.select(col('Partition_id')).distinct().display()

In [0]:
df_flight.groupBy('Partition_id').count().display()

In [0]:
df_flight.filter(col('Partition_id')==1).display()

In [0]:
df_flight.write.format('csv').save('/Volumes/workspace/default/datasets/flight_data_transformed.csv')

In [0]:
spark.conf.get('spark.sql.files.maxPartitionBytes')

In [0]:
134217728/(1024*1024)---default spark partition

In [0]:
df_flight = df_flight.repartition(8)

In [0]:
df_flight = df_flight.withColumn('New_partition_id', spark_partition_id())
df_flight.display()

In [0]:
df_flight.groupBy('New_partition_id').count().display()

In [0]:
df_flight.coalesce(1).write.format('csv').save('/Volumes/workspace/default/datasets/flight_data_transformed_coalesce')

In [0]:
df_flight.select('_metadata').display()

In [0]:
#when to use repartition-- when the distribution of data in imbalanced ...(10000: 100)

In [0]:
#when coalesce is useful----when number of files(part files) is very large and size of each file is very small, then we should apply it.

In [0]:
df_A = spark.range(0,999900099)
#df_A.display()

In [0]:
df_A.count()

In [0]:
df_A.withColumn('partition_id', spark_partition_id()).select('partition_id').distinct().display()

In [0]:
df_A = df_A.withColumn('partition_id', spark_partition_id())
df_A.groupby('partition_id').count().display()

In [0]:
df_A.filter(col('partition_id')==2).display()

In [0]:
df_A = df_A.repartition(16)
df_A.withColumn('partition_id', spark_partition_id()).groupby('partition_id').count().display()

In [0]:
df_A.repartitionById(partitionIdCol='id',numPartitions=10) # range partition
# when we are performing join based on a join key...join key will be the partition id

In [0]:
dept id
1 - 100
2 - 100
3 - 10000
4 - 90
5 - 78

for the above scenario we should avoid using partitionbyID because of dataskewness

In [0]:
partition should be applied on less cardinal columns----
100000 records ---number of partition 100000---high cardinality----emp id in Employee table
100000 records ---number of partiton 10 ---low cardinality-----dept id in employee table

In [0]:
df_A.repartitionByRange(10, col('id'))

In [0]:
df_A.repartition(10,['DEPT_NUM','JOB'])

In [0]:
sales data 


2021  2022  2023  2024  2025
120    170  210   180   200

In [0]:
unpivot--

year sales_val
2021  120
2022  170
2023  210
2024  180
2025  200

In [0]:
df_emp =  spark.read.csv('/Volumes/workspace/default/test_practice/emp.csv',header='True')
df_emp.display()

In [0]:
df_emp.groupBy('DEPTNO').agg(sum('SAL').alias('sum_sal')).display()

In [0]:
df_emp.groupBy().pivot('DEPTNO').agg(sum('SAL').alias('sum_sal')).display()

In [0]:
df_emp.groupBy('DEPTNO').pivot(agg(sum('SAL').alias('sum_sal'))).display()

## Unpivot a DF

In [0]:
df_x = spark.createDataFrame([('1633',24,25,45,56,43),('1636',22,25,40,66,63),('1639',12,15,33,26,43),('1640',34,25,25,46,53)],schema=['prod_id','2021','2022','2023','2024','2025'])
df_x.display()

In [0]:

unpivot_expr = "stack(5, '2021', `2021`, '2022', `2022`, '2023', `2023`, '2024', `2024`, '2025', `2025`) as (Year, revenue)"
df_x_unpivot = df_x.select('prod_id', expr(unpivot_expr))
display(df_x_unpivot)

In [0]:
df_x.melt(ids=['prod_id'],values=['2021','2022','2023','2024','2025'],variableColumnName='Year',valueColumnName='Revenue').display()

In [0]:
variable_cols = [col for col in df_x.columns if col!='prod_id']
variable_cols

In [0]:
df_x.melt(ids=['prod_id'],values=variable_cols,variableColumnName='Year',valueColumnName='Revenue').display()

In [0]:
df_x.melt('prod_id',variableColumnName='Year',valueColumnName='revenue').display()

## Handle Null

In [0]:
df_x.withColumn('2021',when(col('2021')==22 , lit(None)).otherwise(col("2021"))).display()
#.withColumn('2023',when(col('2023')==33 ,lit('')).otherwise(col('2023')))

In [0]:
df_x = df_x.withColumn('2023',col('2023').cast(StringType())).withColumn('2023',when(col('2023')==33 ,lit('')).otherwise(col('2023'))).fillna('0')
df_x.display()


In [0]:
df_x.withColumn('2023',when(col("2023") == '' , lit(None)).otherwise(col("2023"))).display()

In [0]:
df_x.fillna('0').display()

In [0]:
df_emp =  spark.read.csv('/Volumes/workspace/default/test_practice/emp.csv',header='True')
df_emp.display()

In [0]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [0]:
df_emp = df_emp.withColumn('HIREDATE', to_date(col('HIREDATE'), 'dd-MMM-yy'))
df_emp.display()

In [0]:
df_emp.withColumn('tot_exp', round((date_diff(current_date() , col('HIREDATE')))/365 , 2)).display()

In [0]:
df_emp.withColumn('tot_mon', (months_between(current_date() , col('HIREDATE')))).display()

## https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html

In [0]:
%run ./MultiPly_x

In [0]:
print(y)

In [0]:
#file read - separate Notebook
#data transformation - separate Notebook
#write to table/file - separate notebook
master notebook--